In [22]:
# ==========================================================
# CELL 1: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Repo sudah ada, menarik update terbaru...
Already up to date.

✅ Setup selesai. Working dir: /content/skripsi-corn-label-noise


In [ ]:
# ==========================================================
# CELL 2: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

✅ Dependencies terpasang.


In [ ]:
# ==========================================================
# CELL 2.5 (BARU): IMPORT UMUM — dipakai di banyak cell berikutnya
# ==========================================================
import os
import pandas as pd
import numpy as np

In [ ]:
# ==========================================================
# CELL 3: SCRAPING GOOGLE PLAY STORE
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA SEUMUR PROYEK. Setelah cell ini selesai,
# all_reviews_master.csv DIKUNCI -- jangan dijalankan ulang, supaya
# seluruh eksperimen berikutnya memakai sumber data yang identik.
#
# Kalau sempat terputus di tengah jalan, JALANKAN ULANG cell ini --
# scraper akan resume otomatis dari checkpoint terakhir per (app, rating),
# bukan mulai dari nol.
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
 GOOGLE PLAY SCRAPER — STRATIFIED PER RATING 

📡 SeaBank | rating=1 | target=800
   +200 ulasan (total: 200/800)
   +200 ulasan (total: 400/800)
   +200 ulasan (total: 600/800)
   💾 Checkpoint tersimpan (600 baris)
   +200 ulasan (total: 800/800)
✅ Selesai: SeaBank rating=1 -> 800 ulasan bersih.

📡 SeaBank | rating=2 | target=500
   +200 ulasan (total: 200/500)
   +200 ulasan (total: 400/500)
   +200 ulasan (total: 600/500)
   💾 Checkpoint tersimpan (600 baris)
✅ Selesai: SeaBank rating=2 -> 600 ulasan bersih.

📡 SeaBank | rating=3 | target=500
   +200 ulasan (total: 200/500)
   +200 ulasan (total: 400/500)
   +200 ulasan (total: 600/500)
   💾 Checkpoint tersimpan (600 baris)
✅ Selesai: SeaBank rating=3 -> 600 ulasan bersih.

📡 SeaBank | rating=4 | target=700
   +200 ulasan (total: 200/700)
   +200 ulasan (total: 400/700)
   +200 ulasan (total: 600/700)
   💾 Checkpoint tersimpan (600 baris)


In [ ]:
# ==========================================================
# CELL 4: PREPROCESSING + TRAIN-TEST SPLIT (SUMBER KEBENARAN TUNGGAL)
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA. Split ini (train_raw / test) DIKUNCI dan dipakai
# di SELURUH eksperimen berikutnya -- termasuk pilot study proxy 0-4.
# Menjalankan ulang cell ini akan mengganti split yang sudah ada; jangan
# lakukan kecuali kamu sengaja ingin memulai dari nol.
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print("✅ Split train/test sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(" PREPROCESSING ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

⬇️  Mengunduh Kamus Alay (Salsabila dkk., 2018) dari https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv ...
✅ Tersimpan di cache: /content/drive/MyDrive/SKRIPSI_CORN/lexicon/slang_base.csv
📖 Kamus slang dasar: 15007 entri (Salsabila dkk., 2018)
 PREPROCESSING 
📥 Membaca data mentah dari: /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv
📏 Jumlah data awal: 10800 baris
🧹 Cleaning teks (lowercase, URL/tag, emoji, elongasi, slang)...
📊 Cakupan kamus slang: 14/138985 kata (0.01%)
🔍 Teks identik, rating berbeda: 162 kasus
✅ Setelah dibersihkan: 8947 baris (terbuang: 1853)
💾 Disimpan di: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean.csv

 SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) 
✅ Train: 7157 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw.csv
✅ Test : 1790 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test.csv


In [ ]:
# ==========================================================
# CELL 5: PILOT STUDY -- ABLASI PROXY 0-4
# ==========================================================
# Menjalankan clean.py untuk KELIMA metode proxy secara berurutan.
# Hasilnya terkumpul otomatis di results/proxy_ablation_table.csv
# (dipakai untuk narasi pilot study di Bab 1).
#
# CATATAN: proxy 2, 3 butuh fine-tuning K-Fold (5 fold x beberapa epoch),
# jadi cell ini bisa makan waktu cukup lama untuk kelimanya. Kalau kamu
# sudah yakin final proxy = 3 (finetuned_corn) dan hanya ingin lihat
# pilot study SEKALI dan sudah tahu hasilnya, cell ini boleh dilewati --
# langsung ke Cell 6.
#
# PROXY_ID 4 (fusion) akan raise NotImplementedError -- beri tahu saya
# kalau kamu mau lanjut mengimplementasikannya nanti.
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

PILOT_PROXY_IDS = [0, 1, 2, 3]  # 4 (fusion) belum diimplementasikan penuh

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} \n{'='*70}")

    config.set_proxy(pid)

    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table)


 PILOT STUDY -- PROXY_ID = 0 
📌 Proxy aktif: [0] frozen_cls_lr — CLS embedding beku + Logistic Regression (P1)
 CONFIDENT LEARNING — proxy aktif: [0] frozen_cls_lr 
📥 Memuat 7157 baris data train.

🧮 Menghitung OOF pred_probs — proxy [0] frozen_cls_lr


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding (cls): 100%|██████████| 448/448 [00:28<00:00, 15.84it/s]



📐 Kualitas proxy [frozen_cls_lr]:
   Exact Accuracy : 0.4234
   MAE            : 0.9462
   Off-by-1 Acc   : 0.7486
   QWK            : 0.6103

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3495 baris diflag (48.83%)
   'prune_by_noise_rate': 2980 baris diflag (41.64%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3495 / sisa 3662
   Severity-aware : buang 1467 / sisa 5690

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2028
2     928
3     432
4     107
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__frozen_cls_lr.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__frozen_cls_lr.csv

📝 Sample validasi manusia (50 baris, stratified by rating_diff):
   /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5000
      Epoch 2/3 - Loss: 0.4358
      Epoch 3/3 - Loss: 0.3685
   [Proxy finetuned_corn] Fold 5/5 (train=5726, val=1431)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5074
      Epoch 2/3 - Loss: 0.4416
      Epoch 3/3 - Loss: 0.3823

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4394
   MAE            : 0.8041
   Off-by-1 Acc   : 0.8223
   QWK            : 0.6692

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3649 baris diflag (50.99%)
   'prune_by_noise_rate': 3288 baris diflag (45.94%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3649 / sisa 3508
   Severity-aware : buang 1145 / sisa 6012

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2504
2     838
3     239
4      68
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn.csv

📝 Sample validasi manusia (50 baris, stra

,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.423362,0.946207,0.748638,0.610310,48.833310
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.422384,0.953752,0.748777,0.598690,47.953053
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.438731,0.801593,0.817242,0.653276,51.893251
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.439430,0.804108,0.822272,0.669240,50.985050


In [ ]:
# ==========================================================
# CELL 6: PROXY FINAL (SESUAI BAB 3) -- HASIL INI YANG DIPAKAI BAB 4
# ==========================================================
# ⚠️ PENTING: restart runtime dulu sebelum cell ini kalau tadi sempat
# jalankan Cell 5 (loop pilot study) -- supaya config bersih, tidak ada
# sisa reload yang bikin path tercampur.
# ==========================================================
from src import config

config.set_proxy(3)
print(f"📌 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
📌 Proxy final terkunci: [3] finetuned_corn
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn 
📥 Memuat 7157 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4394
   MAE            : 0.8041
   Off-by-1 Acc   : 0.8223
   QWK            : 0.6692

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3649 baris diflag (50.99%)
   'prune_by_noise_rate': 3288 baris diflag (45.94%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3649 / sisa 3508
   Severity-aware : buang 1145 / sisa 6012

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2504
2     838
3     239
4      68
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /

In [26]:
# ==========================================================
# CELL 7: VALIDASI MANUSIA — WAJIB SEBELUM LANJUT
# ==========================================================
from src import config

print("⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan cell ini lagi untuk cek kelengkapan + hitung agreement rate.")

⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️
1. Buka: /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_sample.csv
2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:
   'noise' / 'not_noise' / 'ambiguous'
3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).
4. Jalankan cell ini lagi untuk cek kelengkapan + hitung agreement rate.


In [ ]:
# Jalankan sel ini SETELAH selesai isi manual di atas.
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 8 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

In [ ]:
# ==========================================================
# CELL 8: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(" 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆")
print("=" * 100)
display(df_final)

In [ ]:
# ==========================================================
# CELL 9: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI
# ==========================================================
# Berbeda dari 4 percobaan sebelumnya: prediksi dikumpulkan dari
# KETIGA seed (bukan cuma seed 42), diagregasi dulu, baru diuji --
# sesuai janji Subbab 3.9.2 proposal.
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

In [ ]:
# ==========================================================
# CELL 10: RINGKASAN AKHIR -- SIAP DISALIN KE BAB 4
# ==========================================================
from src import config

print("=" * 70)
print(" RINGKASAN LENGKAP UNTUK BAB 4 ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (P0-P4) -- untuk Bab 1 (pilot study):")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    display(pd.read_csv(config.PROXY_QUALITY_LOG_FILE))

print(f"\n[2] KUALITAS PROXY FINAL [{config.PROXY_NAME}] -- untuk Bab 4:")
print(proxy_metrics)

print("\n[3] VALIDASI MANUSIA -- untuk Bab 4:")
print(result)

print("\n[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))

print("\n[5] UJI SIGNIFIKANSI (3 hipotesis pre-registered) -- untuk Bab 4:")
display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))

print("\n[6] EFFECT SIZE + CI 95% -- untuk Bab 4:")
display(effect_sizes)

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)